# Event-VLM Colab Smoke Proof

This notebook is the Colab-first proof lane for Event-VLM. It validates structural feasibility and figure prototypes only; it is not headline benchmark evidence.

Expected outputs:
- `outputs/colab_smoke/figure2_token_mask_overlay.png`
- `outputs/colab_smoke/figure3_fc_sparse_selection.png`
- `outputs/colab_smoke/dense_vs_sparse_latency_tokens.csv`
- `outputs/colab_smoke/report_status_table.csv`
- `outputs/colab_smoke/colab_smoke_summary.json`


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('EVENT_VLM_REPO_URL', 'https://github.com/shlee-hd/Event-VLM.git')
BRANCH = os.environ.get('EVENT_VLM_BRANCH', 'colab-smoke-proof-2026-04-25')
WORKDIR = Path('/content/Event-VLM') if IN_COLAB else Path.cwd().resolve()

if IN_COLAB and not WORKDIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKDIR)
    ], check=True)

os.chdir(WORKDIR)
print('workdir =', WORKDIR)
print('branch =', BRANCH)


In [ ]:
# Minimal dependencies for the smoke lane. Avoid installing full LLaVA here.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pytest>=7.4.0', 'omegaconf>=2.3.0', 'hydra-core>=1.3.0',
    'opencv-python-headless>=4.8.0', 'pillow>=10.0.0', 'scikit-learn>=1.2.0'
], check=True)


In [ ]:
script = WORKDIR / 'scripts' / 'colab_smoke_event_vlm.py'
assert script.exists(), f'Missing smoke script: {script}'

cmd = [sys.executable, str(script), '--output-dir', 'outputs/colab_smoke']
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
summary_path = WORKDIR / 'outputs' / 'colab_smoke' / 'colab_smoke_summary.json'
summary = json.loads(summary_path.read_text())
print(json.dumps({
    'acceptance': summary['acceptance'],
    'metrics': summary['metrics'],
    'artifacts': summary['artifacts'],
}, indent=2))

assert all(summary['acceptance'].values()), summary['acceptance']


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(WORKDIR / 'outputs/colab_smoke/figure2_token_mask_overlay.png')))
display(Image(filename=str(WORKDIR / 'outputs/colab_smoke/figure3_fc_sparse_selection.png')))


In [ ]:
import zipfile

out_dir = WORKDIR / 'outputs' / 'colab_smoke'
zip_path = WORKDIR / 'outputs' / 'colab_smoke_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in out_dir.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(WORKDIR))
print('artifact zip =', zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
